****

**Nombre:** Juan Martín Cusme Carrera

**Materia:** Recuperación de información

**Curso:** GR1CC

**Fecha:** 11/11/2026

****

# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
from pathlib import Path
import sys
import re
import unicodedata
import numpy as np
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

# Reutilizar las dependencias preparadas en la práctica anterior.
sys.path.insert(0, str(raiz / ".codex-runtime"))
# Si hacen falta en otro equipo: %pip install nltk numpy pandas scikit-learn
import nltk
import sklearn

nltk.data.path.insert(0, str(raiz / "datos_booleansearch" / "nltk_data"))
from nltk.corpus import gutenberg

def normalizar(texto):
    texto = unicodedata.normalize("NFC", texto).lower()
    return re.findall(r"[^\W\d_]+", texto)

carpeta_resultados = raiz / "datos_vectorsearch"
carpeta_resultados.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 230)
print("Versión de scikit-learn:", sklearn.__version__)
print("Ejemplo de normalización:", normalizar("¡Árbol! Don't stop: 2026."))

Versión de scikit-learn: 1.7.2
Ejemplo de normalización: ['árbol', 'don', 't', 'stop']


In [2]:
ids_documentos = gutenberg.fileids()
textos_originales = {}
corpus = {}
titulos = {}
vocabulario_corpus = set()
filas_corpus = []

for id_doc in ids_documentos:
    texto = gutenberg.raw(id_doc)
    tokens = normalizar(texto)
    encabezado = texto.splitlines()[0].strip().strip("[]")
    titulo = encabezado.split(" by ")[0].rstrip(",")
    textos_originales[id_doc] = texto
    corpus[id_doc] = tokens
    titulos[id_doc] = titulo
    vocabulario_corpus.update(tokens)
    filas_corpus.append([id_doc, titulo, len(set(tokens)), len(tokens)])

tabla_corpus = pd.DataFrame(filas_corpus, columns=["Identificador", "Título", "Términos únicos", "Longitud (tokens)"])
N = len(corpus)
longitudes = tabla_corpus["Longitud (tokens)"].to_numpy()
assert N == 18
print("Número de documentos:", N)
print("Tamaño del vocabulario:", len(vocabulario_corpus))
print("Longitud mínima:", longitudes.min())
print("Longitud media:", round(longitudes.mean(), 2))
print("Longitud máxima:", longitudes.max())
display(tabla_corpus)
tabla_corpus.to_csv(carpeta_resultados / "corpus.csv", index=False)

Número de documentos: 18
Tamaño del vocabulario: 41509
Longitud mínima: 6934
Longitud media: 118670.78
Longitud máxima: 791842


,Identificador,Título,Términos únicos,Longitud (tokens)
0,austen-emma.txt,Emma,7096,161975
1,austen-persuasion.txt,Persuasion,5740,84123
2,austen-sense.txt,Sense and Sensibility,6283,120734
3,bible-kjv.txt,The King James Bible,12568,791842
4,blake-poems.txt,Poems,1510,6934
5,bryant-stories.txt,Stories to Tell to Children,3834,46697
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1521,16353
7,carroll-alice.txt,Alice's Adventures in Wonderland,2569,27335
8,chesterton-ball.txt,The Ball and The Cross,8176,82785
9,chesterton-brown.txt,The Wisdom of Father Brown,7737,73286


La longitud y la cantidad de términos únicos no son lo mismo. Un documento puede
repetir muchas veces las mismas palabras. Los títulos se toman del encabezado
original de cada archivo; el encabezado también forma parte de los tokens,
igual que en la práctica anterior.

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

## Funciones de TF, DF, IDF y peso TF-IDF

TF cuenta las apariciones de una palabra dentro de un documento. DF cuenta en
cuántos documentos aparece, por lo que una palabra repetida muchas veces en un
libro solo aporta uno a su DF. IDF se calcula sobre los 18 documentos:
$ln(N/DF)+1$.

Para mostrar los seis pesos solicitados se eligen tres términos (whale, ship
y captain) en dos libros: Moby Dick y Persuasion. Son seis pares
término-documento, todos calculados con la fórmula indicada en el enunciado.

In [3]:
def calcular_tf(tokens):
    frecuencias = {}
    for termino in tokens:
        if termino not in frecuencias:
            frecuencias[termino] = 0
        frecuencias[termino] += 1
    return frecuencias

def calcular_df(documentos):
    frecuencias = {}
    for tokens in documentos.values():
        for termino in set(tokens):
            if termino not in frecuencias:
                frecuencias[termino] = 0
            frecuencias[termino] += 1
    return frecuencias

def calcular_idf(frecuencias_df, cantidad_documentos):
    pesos = {}
    for termino, frecuencia in frecuencias_df.items():
        pesos[termino] = np.log(cantidad_documentos / frecuencia) + 1
    return pesos

def calcular_peso_tfidf(termino, frecuencias_tf, pesos_idf):
    return frecuencias_tf.get(termino, 0) * pesos_idf.get(termino, 0)

In [4]:
tf_documentos = {}
for id_doc, tokens in corpus.items():
    tf_documentos[id_doc] = calcular_tf(tokens)

df = calcular_df(corpus)
idf = calcular_idf(df, N)
filas_pesos = []

for id_doc in ["melville-moby_dick.txt", "austen-persuasion.txt"]:
    for termino in ["whale", "ship", "captain"]:
        frecuencia = tf_documentos[id_doc].get(termino, 0)
        peso = calcular_peso_tfidf(termino, tf_documentos[id_doc], idf)
        filas_pesos.append([id_doc, termino, frecuencia, df[termino], idf[termino], peso])

tabla_seis_pesos = pd.DataFrame(filas_pesos, columns=["Documento", "Término", "TF", "DF del corpus", "IDF", "TF-IDF"])
display(tabla_seis_pesos.round(6))
print("DF de 'the':", df["the"])
print("IDF de 'the':", idf["the"])
tabla_seis_pesos.to_csv(carpeta_resultados / "seis_pesos.csv", index=False)

,Documento,Término,TF,DF del corpus,IDF,TF-IDF
0,melville-moby_dick.txt,whale,1226,6,2.098612,2572.898666
1,melville-moby_dick.txt,ship,518,13,1.325422,686.568803
2,melville-moby_dick.txt,captain,329,9,1.693147,557.045422
3,austen-persuasion.txt,whale,0,6,2.098612,0.000000
4,austen-persuasion.txt,ship,6,13,1.325422,7.952534
5,austen-persuasion.txt,captain,303,9,1.693147,513.023596


DF de 'the': 18
IDF de 'the': 1.0


La palabra "the" aparece en todos los documentos, por eso su IDF es 1 con esta
fórmula. Las palabras presentes en menos documentos obtienen un IDF mayor.
Si un término no aparece en un documento, su TF y su peso en ese documento son
cero. Los términos ajenos al vocabulario se ignoran al representar una consulta.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

## Matriz documento-término calculada explícitamente

El vocabulario se ordena alfabéticamente. Cada fila corresponde a un documento
y cada columna a una palabra en ese orden. Primero se llena la matriz de
conteos; después se multiplica cada columna por su IDF. Se comprueba que el DF
obtenido con la función coincide con el conteo de filas no nulas de cada columna.

In [5]:
vocabulario = sorted(vocabulario_corpus)
posiciones = {termino: i for i, termino in enumerate(vocabulario)}
matriz_tf = np.zeros((N, len(vocabulario)))

for i, id_doc in enumerate(ids_documentos):
    for termino, frecuencia in tf_documentos[id_doc].items():
        j = posiciones[termino]
        matriz_tf[i, j] = frecuencia

vector_df = np.array([df[termino] for termino in vocabulario])
vector_idf = np.array([idf[termino] for termino in vocabulario])
matriz_tfidf = matriz_tf * vector_idf

assert np.array_equal(vector_df, (matriz_tf > 0).sum(axis=0))
assert np.array_equal(matriz_tf.sum(axis=1), longitudes)
print("Dimensiones de TF:", matriz_tf.shape)
print("Dimensiones de TF-IDF:", matriz_tfidf.shape)

terminos_ejemplo = ["alice", "captain", "hamlet", "love", "the", "whale"]
columnas_ejemplo = [posiciones[termino] for termino in terminos_ejemplo]
print("Ejemplo de la matriz TF-IDF:")
display(pd.DataFrame(matriz_tfidf[:, columnas_ejemplo], index=ids_documentos, columns=terminos_ejemplo).round(3))

Dimensiones de TF: (18, 41509)
Dimensiones de TF-IDF: (18, 41509)
Ejemplo de la matriz TF-IDF:


,alice,captain,hamlet,love,the,whale
austen-emma.txt,0.000,5.079,0.000,138.332,5204.0,0.000
austen-persuasion.txt,0.000,513.024,0.000,49.658,3329.0,0.000
austen-sense.txt,0.000,0.000,2.099,91.039,4105.0,0.000
bible-kjv.txt,0.000,235.347,0.000,367.702,64023.0,6.296
blake-poems.txt,0.000,0.000,0.000,34.287,439.0,0.000
bryant-stories.txt,0.000,3.386,0.000,15.370,3452.0,33.578
burgess-busterbrown.txt,0.000,0.000,0.000,1.182,660.0,0.000
carroll-alice.txt,1111.120,0.000,0.000,3.547,1642.0,0.000
chesterton-ball.txt,0.000,0.000,4.197,29.558,4981.0,4.197
chesterton-brown.txt,0.000,32.170,0.000,10.641,4670.0,0.000


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

## Verificación con scikit-learn

Se entregan directamente las listas de tokens al vectorizador. La función
"devolver_tokens" evita que se tokenicen otra vez; así también se conservan las
palabras de una sola letra. El vocabulario fija el mismo orden de columnas.

Se establece una tolerancia absoluta de **1e-10**, sin tolerancia relativa.
Se compara tanto el IDF como la matriz completa de pesos sin normalizar.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

def devolver_tokens(tokens):
    return tokens

vectorizador = TfidfVectorizer(
    analyzer=devolver_tokens,
    vocabulary=posiciones,
    lowercase=False,
    token_pattern=None,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)

listas_tokens = [corpus[id_doc] for id_doc in ids_documentos]
matriz_sklearn = vectorizador.fit_transform(listas_tokens)
matriz_libreria = matriz_sklearn.toarray()
tolerancia = 1e-10
diferencia_idf = np.max(np.abs(vector_idf - vectorizador.idf_))
diferencia_tfidf = np.max(np.abs(matriz_tfidf - matriz_libreria))

assert list(vectorizador.get_feature_names_out()) == vocabulario
assert np.allclose(vector_idf, vectorizador.idf_, atol=tolerancia, rtol=0)
assert np.allclose(matriz_tfidf, matriz_libreria, atol=tolerancia, rtol=0)
print("Máxima diferencia absoluta de IDF:", diferencia_idf)
print("Máxima diferencia absoluta de TF-IDF:", diferencia_tfidf)
print("Las matrices coinciden dentro de la tolerancia:", diferencia_tfidf <= tolerancia)

Máxima diferencia absoluta de IDF: 0.0
Máxima diferencia absoluta de TF-IDF: 0.0
Las matrices coinciden dentro de la tolerancia: True


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

## Cuatro necesidades de información

Se usan consultas en inglés porque los libros de este corpus están en inglés.
Cada consulta busca un tema diferente. Se representa con el vocabulario y los
IDF ya calculados y no se vuelve a ajustar el vectorizador con las consultas.

In [7]:
consultas = [
    ("C1: mar", "whale ship captain", "Encontrar relatos sobre ballenas, barcos y capitanes."),
    ("C2: fantasía", "alice rabbit queen", "Encontrar aventuras de Alicia con el conejo y la reina."),
    ("C3: familia", "love marriage family", "Encontrar historias sobre amor, matrimonio y relaciones familiares."),
    ("C4: Hamlet", "hamlet ghost revenge", "Encontrar textos sobre Hamlet, el fantasma y la venganza.")
]

tabla_consultas = pd.DataFrame(consultas, columns=["Consulta", "Texto", "Necesidad de información"])
tokens_consultas = [normalizar(texto) for nombre, texto, necesidad in consultas]
display(tabla_consultas)

for consulta, tokens in zip(consultas, tokens_consultas):
    desconocidos = [termino for termino in tokens if termino not in posiciones]
    print(consulta[0], "- Tokens:", tokens, "- Fuera del vocabulario:", desconocidos)

,Consulta,Texto,Necesidad de información
0,C1: mar,whale ship captain,"Encontrar relatos sobre ballenas, barcos y capitanes."
1,C2: fantasía,alice rabbit queen,Encontrar aventuras de Alicia con el conejo y la reina.
2,C3: familia,love marriage family,"Encontrar historias sobre amor, matrimonio y relaciones familiares."
3,C4: Hamlet,hamlet ghost revenge,"Encontrar textos sobre Hamlet, el fantasma y la venganza."


C1: mar - Tokens: ['whale', 'ship', 'captain'] - Fuera del vocabulario: []
C2: fantasía - Tokens: ['alice', 'rabbit', 'queen'] - Fuera del vocabulario: []
C3: familia - Tokens: ['love', 'marriage', 'family'] - Fuera del vocabulario: []
C4: Hamlet - Tokens: ['hamlet', 'ghost', 'revenge'] - Fuera del vocabulario: []


## Vectores de consulta y similitud coseno explícita

Se calcula el producto punto y se divide por el producto de las magnitudes de
los dos vectores completos. Si alguno tiene norma cero, se devuelve cero para
evitar una división por cero. La biblioteca utiliza el vectorizador ajustado
mediante "transform" y después "cosine_similarity".

In [8]:
def vectorizar_consulta(tokens):
    frecuencias = calcular_tf(tokens)
    vector = np.zeros(len(vocabulario))
    for termino in frecuencias:
        if termino in posiciones:
            vector[posiciones[termino]] = calcular_peso_tfidf(termino, frecuencias, idf)
    return vector

def similitud_coseno(vector_q, vector_d):
    producto = np.dot(vector_q, vector_d)
    norma_q = np.sqrt(np.sum(vector_q ** 2))
    norma_d = np.sqrt(np.sum(vector_d ** 2))
    if norma_q == 0 or norma_d == 0:
        return 0.0
    return producto / (norma_q * norma_d)

vectores_consultas = np.array([vectorizar_consulta(tokens) for tokens in tokens_consultas])
consultas_sklearn = vectorizador.transform(tokens_consultas)
diferencia_consultas = np.max(np.abs(vectores_consultas - consultas_sklearn.toarray()))
assert np.allclose(vectores_consultas, consultas_sklearn.toarray(), atol=tolerancia, rtol=0)
print("Dimensiones de las consultas:", vectores_consultas.shape)
print("Máxima diferencia absoluta de los vectores de consulta:", diferencia_consultas)

Dimensiones de las consultas: (4, 41509)
Máxima diferencia absoluta de los vectores de consulta: 0.0


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similitudes_explicitas = np.zeros((len(consultas), N))
for i, vector_q in enumerate(vectores_consultas):
    for j, vector_d in enumerate(matriz_tfidf):
        similitudes_explicitas[i, j] = similitud_coseno(vector_q, vector_d)

similitudes_sklearn = cosine_similarity(consultas_sklearn, matriz_sklearn)
diferencia_coseno = np.max(np.abs(similitudes_explicitas - similitudes_sklearn))
assert np.allclose(similitudes_explicitas, similitudes_sklearn, atol=tolerancia, rtol=0)
print("Máxima diferencia absoluta de las similitudes:", diferencia_coseno)
print("Las similitudes coinciden:", diferencia_coseno <= tolerancia)

tabla_similitudes = pd.DataFrame(similitudes_explicitas, index=[consulta[0] for consulta in consultas], columns=ids_documentos)
display(tabla_similitudes.round(6))
tabla_similitudes.to_csv(carpeta_resultados / "similitudes_tfidf.csv")

Máxima diferencia absoluta de las similitudes: 2.6367796834847468e-15
Las similitudes coinciden: True


,austen-emma.txt,austen-persuasion.txt,austen-sense.txt,bible-kjv.txt,blake-poems.txt,bryant-stories.txt,burgess-busterbrown.txt,carroll-alice.txt,chesterton-ball.txt,chesterton-brown.txt,chesterton-thursday.txt,edgeworth-parents.txt,melville-moby_dick.txt,milton-paradise.txt,shakespeare-caesar.txt,shakespeare-hamlet.txt,shakespeare-macbeth.txt,whitman-leaves.txt
C1: mar,0.000225,0.036287,0.000000,0.001728,0.000000,0.005457,0.000000,0.000000,0.002942,0.003888,0.000498,0.002744,0.112065,0.000086,0.000000,0.001486,0.000346,0.005125
C2: fantasía,0.000025,0.000047,0.000033,0.000198,0.002641,0.002743,0.007133,0.310257,0.000736,0.000640,0.000658,0.000757,0.000317,0.000280,0.000000,0.001099,0.000000,0.000054
C3: familia,0.010585,0.013830,0.013032,0.002942,0.026155,0.002764,0.000335,0.000876,0.002259,0.005700,0.001990,0.005226,0.001201,0.009994,0.000415,0.001896,0.000000,0.012918
C4: Hamlet,0.000000,0.000125,0.000293,0.000980,0.000000,0.000195,0.000000,0.000000,0.000524,0.000479,0.000143,0.000585,0.000971,0.003434,0.002520,0.055856,0.001566,0.000431


## Ejemplo detallado: consulta sobre el mar y Moby Dick

Se muestran las palabras compartidas, sus dos pesos y el producto parcial de
cada una. La suma de esos productos debe ser igual al producto punto. La norma
del documento considera todo su vocabulario, incluyendo las palabras que
no están en la consulta.

In [10]:
id_ejemplo = "melville-moby_dick.txt"
fila_ejemplo = ids_documentos.index(id_ejemplo)
vector_q = vectores_consultas[0]
vector_d = matriz_tfidf[fila_ejemplo]
compartidos = np.where((vector_q > 0) & (vector_d > 0))[0]
filas_productos = []

for j in compartidos:
    termino = vocabulario[j]
    filas_productos.append([termino, vector_q[j], vector_d[j], vector_q[j] * vector_d[j]])

tabla_productos = pd.DataFrame(filas_productos, columns=["Término compartido", "Peso consulta", "Peso documento", "Producto parcial"])
producto_punto = np.dot(vector_q, vector_d)
norma_q = np.sqrt(np.sum(vector_q ** 2))
norma_d = np.sqrt(np.sum(vector_d ** 2))
coseno_ejemplo = similitud_coseno(vector_q, vector_d)
assert np.isclose(tabla_productos["Producto parcial"].sum(), producto_punto, atol=tolerancia, rtol=0)

display(tabla_productos.round(6))
display(pd.DataFrame([[producto_punto, norma_q, norma_d, coseno_ejemplo]], columns=["Producto punto", "Norma consulta", "Norma documento", "Similitud coseno"]).round(6))
tabla_productos.to_csv(carpeta_resultados / "producto_punto.csv", index=False)

,Término compartido,Peso consulta,Peso documento,Producto parcial
0,captain,1.693147,557.045422,943.159886
1,ship,1.325422,686.568803,909.993671
2,whale,2.098612,2572.898666,5399.516758


,Producto punto,Norma consulta,Norma documento,Similitud coseno
0,7252.670316,3.004607,21539.783392,0.112065


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

## Top-5 por similitud coseno

Los documentos se ordenan por puntuación descendente. Si hay empate se usa el
identificador en orden alfabético. El fragmento se toma del texto original
alrededor de una palabra de la consulta; se juntan los saltos de línea para
facilitar la lectura. Se compara el top-5 explícito con el de la biblioteca.

In [11]:
def obtener_fragmento(texto, tokens_consulta):
    terminos_consulta = set(tokens_consulta)
    inicio = 0
    for coincidencia in re.finditer(r"[^\W\d_]+", texto):
        termino = normalizar(coincidencia.group())[0]
        if termino in terminos_consulta:
            inicio = max(0, coincidencia.start() - 60)
            break
    fin = min(len(texto), inicio + 220)
    fragmento = re.sub(r"\s+", " ", texto[inicio:fin]).strip()
    if inicio > 0:
        fragmento = "..." + fragmento
    if fin < len(texto):
        fragmento += "..."
    return fragmento

def crear_ranking(puntuaciones, tokens_consulta):
    tabla = pd.DataFrame({
        "Identificador": ids_documentos,
        "Título": [titulos[id_doc] for id_doc in ids_documentos],
        "Puntuación": puntuaciones
    })
    tabla = tabla.sort_values(["Puntuación", "Identificador"], ascending=[False, True])
    tabla = tabla.head(5).reset_index(drop=True)
    tabla.insert(0, "Posición", range(1, len(tabla) + 1))
    tabla["Fragmento"] = [obtener_fragmento(textos_originales[id_doc], tokens_consulta) for id_doc in tabla["Identificador"]]
    return tabla

In [12]:
rankings_tfidf = {}
for i, (nombre, texto, necesidad) in enumerate(consultas):
    ranking = crear_ranking(similitudes_explicitas[i], tokens_consultas[i])
    ranking_libreria = crear_ranking(similitudes_sklearn[i], tokens_consultas[i])
    assert ranking["Identificador"].tolist() == ranking_libreria["Identificador"].tolist()
    assert np.allclose(ranking["Puntuación"], ranking_libreria["Puntuación"], atol=tolerancia, rtol=0)
    rankings_tfidf[nombre] = ranking
    print(nombre, "-", texto)
    display(ranking.round({"Puntuación": 6}))
    ranking.to_csv(carpeta_resultados / ("ranking_tfidf_" + str(i + 1) + ".csv"), index=False)

C1: mar - whale ship captain


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.112065,"...in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the letter H, which almost alone maketh the signification of the word, you deliver th..."
1,2,austen-persuasion.txt,Persuasion,0.036287,"...e of Monkford, however suspicious appearances may be, but a Captain Frederick Wentworth, his brother, who being made commander in consequence of the action off St Domingo, and not immediately employed, had come into Some..."
2,3,bryant-stories.txt,Stories to Tell to Children,0.005457,"...icked him out again. HOW BROTHER RABBIT FOOLED THE WHALE AND THE ELEPHANT One day little Brother Rabbit was running along on the sand, lippety, lippety, when he saw the Whale and the Elephant talking toget..."
3,4,whitman-leaves.txt,Leaves of Grass,0.005125,"...w not whither, yet ever full of faith, Consort to every ship that sails, sail you! Bear forth to them folded my love, (dear mariners, for you I fold it here in every leaf;) Speed on my book! spread your white sai..."
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.003888,"...o alter the equilibrium; the whole coach heeled over like a ship and crashed through the fringe of bushes over the cliff. Muscari threw an arm round Ethel, who clung to him, and shouted aloud. It was for such moments t..."


C2: fantasía - alice rabbit queen


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,carroll-alice.txt,Alice's Adventures in Wonderland,0.310257,"[Alice's Adventures in Wonderland by Lewis Carroll 1865] CHAPTER I. Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had pe..."
1,2,burgess-busterbrown.txt,The Adventures of Buster Bear,0.007133,...He felt that he had the greatest news to tell since Peter Rabbit had first found the tracks of Buster Bear in the Green Forest. He couldn't keep it to himself a minute longer than he had to. So he hurried to the Smil...
2,3,bryant-stories.txt,Stories to Tell to Children,0.002743,"...unless somebody picked him out again. HOW BROTHER RABBIT FOOLED THE WHALE AND THE ELEPHANT One day little Brother Rabbit was running along on the sand, lippety, lippety, when he saw the Whale and the Elep..."
3,4,blake-poems.txt,Poems,0.002641,"...dreamt a dream! What can it mean? And that I was a maiden Queen Guarded by an Angel mild: Witless woe was ne'er beguiled! And I wept both night and day, And he wiped my tears away; And I wept both day and night,..."
4,5,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.001099,"...on you? Ham. Not so my Lord, I am too much i'th' Sun Queen. Good Hamlet cast thy nightly colour off, And let thine eye looke like a Friend on Denmarke. Do not for euer with thy veyled lids Seeke for thy Noble Fath..."


C3: familia - love marriage family


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,blake-poems.txt,Poems,0.026155,"...h a little space, That we may learn to bear the beams of love And these black bodies and this sunburnt face Is but a cloud, and like a shady grove. ""For when our souls have learn'd the heat to bear, The clou..."
1,2,austen-persuasion.txt,Persuasion,0.013830,"...proved it by adding, for the information of himself and his family, these words, after the date of Mary's birth-- ""Married, December 16, 1810, Charles, son and heir of Charles Musgrove, Esq. of Uppercross, in the county..."
2,3,austen-sense.txt,Sense and Sensibility,0.013032,"...Sense and Sensibility by Jane Austen 1811] CHAPTER 1 The family of Dashwood had long been settled in Sussex. Their estate was large, and their residence was at Norland Park, in the centre of their property, where, for..."
3,4,whitman-leaves.txt,Leaves of Grass,0.012918,"...ery ship that sails, sail you! Bear forth to them folded my love, (dear mariners, for you I fold it here in every leaf;) Speed on my book! spread your white sails my little bark athwart the imperious waves, Chant..."
4,5,austen-emma.txt,Emma,0.010585,"..., indulgent father; and had, in consequence of her sister's marriage, been mistress of his house from a very early period. Her mother had died too long ago for her to have more than an indistinct remembrance of her care..."


C4: Hamlet

 - hamlet ghost revenge


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.055856,[The Tragedie of Hamlet by William Shakespeare 1599] Actus Primus. Scoena Prima. Enter Barnardo and Francisco two Centinels. Barnardo. Who's there? Fran. Nay answer me: Stand & vnfold your selfe Bar. Long liu...
1,2,milton-paradise.txt,Paradise Lost,0.003434,"...l Serpent; he it was whose guile, Stirred up with envy and revenge, deceived The mother of mankind, what time his pride Had cast him out from Heaven, with all his host Of rebel Angels, by whose aid, aspiring To set..."
2,3,shakespeare-caesar.txt,The Tragedie of Julius Caesar,0.002520,...downe Where I left reading? Heere it is I thinke. Enter the Ghost of Caesar. How ill this Taper burnes. Ha! Who comes heere? I thinke it is the weakenesse of mine eyes That shapes this monstrous Apparition. It comes vpo...
3,4,shakespeare-macbeth.txt,The Tragedie of Macbeth,0.001566,"...Tarquins rauishing sides, towards his designe Moues like a Ghost. Thou sowre and firme-set Earth Heare not my steps, which they may walke, for feare Thy very stones prate of my where-about, And take the present horror f..."
4,5,bible-kjv.txt,The King James Bible,0.000980,"...hreescore and fifteen years. 25:8 Then Abraham gave up the ghost, and died in a good old age, an old man, and full of years; and was gathered to his people. 25:9 And his sons Isaac and Ishmael buried him in the cave of..."


La coincidencia de ambas implementaciones comprueba que usan los mismos pesos y
el mismo cálculo de coseno. Esto no demuestra por sí solo que todos los
resultados sean relevantes para el usuario. La recuperación usa coincidencias
de palabras y no interpreta sinónimos ni el significado completo de los textos.

## Comprobaciones de consultas sin coincidencias

Una consulta vacía o compuesta por términos desconocidos debe producir un vector
nulo y puntuación cero, sin cambiar el vocabulario ni los IDF aprendidos.

In [13]:
for texto_prueba in ["", "qzxqzxqzxausente"]:
    tokens_prueba = normalizar(texto_prueba)
    vector_prueba = vectorizar_consulta(tokens_prueba)
    vector_prueba_sklearn = vectorizador.transform([tokens_prueba])
    assert np.all(vector_prueba == 0)
    assert np.all(vector_prueba_sklearn.toarray() == 0)
    assert np.all(cosine_similarity(vector_prueba_sklearn, matriz_sklearn) == 0)
    for vector_documento in matriz_tfidf:
        assert similitud_coseno(vector_prueba, vector_documento) == 0

print("Consulta vacía y consulta inexistente: puntuaciones cero en el cálculo explícito y en scikit-learn.")
assert np.array_equal(vector_idf, np.array([idf[termino] for termino in vocabulario]))
assert np.allclose(vector_idf, vectorizador.idf_, atol=tolerancia, rtol=0)

Consulta vacía y consulta inexistente: puntuaciones cero en el cálculo explícito y en scikit-learn.


In [14]:
tabla_verificacion = pd.DataFrame([
    ["IDF", diferencia_idf, diferencia_idf <= tolerancia],
    ["Matriz TF-IDF", diferencia_tfidf, diferencia_tfidf <= tolerancia],
    ["Vectores de consulta", diferencia_consultas, diferencia_consultas <= tolerancia],
    ["Similitud coseno", diferencia_coseno, diferencia_coseno <= tolerancia]
], columns=["Comparación", "Máxima diferencia absoluta", "Dentro de tolerancia"])
display(tabla_verificacion)
tabla_verificacion.to_csv(carpeta_resultados / "verificacion.csv", index=False)

,Comparación,Máxima diferencia absoluta,Dentro de tolerancia
0,IDF,0.000000e+00,True
1,Matriz TF-IDF,0.000000e+00,True
2,Vectores de consulta,0.000000e+00,True
3,Similitud coseno,2.636780e-15,True


## Resultados observados

Se trabajó con 18 documentos y 41.509 términos distintos. Las longitudes fueron de 6.934 a 791.842 tokens, con una media de 118.670,78. Cada término conserva el mismo IDF para los documentos y las consultas.

La diferencia máxima de los IDF, los pesos TF-IDF y los vectores de consulta fue 0. Para el coseno fue 2,637x10^-15, dentro de la tolerancia absoluta de 1e-10. Las dos implementaciones produjeron el mismo orden y puntuaciones dentro de esa tolerancia en los cuatro top-5.

Los primeros resultados para las consultas del mar, fantasía y Hamlet fueron Moby Dick, Alice's Adventures in Wonderland y The Tragedie of Hamlet, respectivamente. Para la consulta sobre amor, matrimonio y familia, Poems, de William Blake, quedó primero. Los fragmentos permiten revisar el contexto de las palabras que contribuyeron a cada resultado.